In [ ]:
# Cell 1: repo bootstrap + shared runtime setup.
import hashlib
import importlib
import json
import os
import sys
import time
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "browser").exists():
    ROOT = ROOT.parent.resolve()
if not (ROOT / "browser").exists():
    raise RuntimeError("Could not locate repo root.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import browser.driver as browser_driver
import browser.linkedin as linkedin
import core.config as core_config
import storage.embedded_mongo as embedded_mongo
import tasks.candidate_scoring_task as candidate_scoring_task
import tasks.detail_scoring_task as detail_scoring_task
import tasks.onboarding_task as onboarding_task
import tests.llm_test_linkedin.linkedin_runtime as linkedin_runtime
import tests.llm_test_linkedin.query_planner_runtime as query_planner_runtime
from shared.runtime_config import build_candidate_pipeline_context, build_pipeline_runtime_config

browser_driver = importlib.reload(browser_driver)
linkedin = importlib.reload(linkedin)
core_config = importlib.reload(core_config)
embedded_mongo = importlib.reload(embedded_mongo)
candidate_scoring_task = importlib.reload(candidate_scoring_task)
detail_scoring_task = importlib.reload(detail_scoring_task)
onboarding_task = importlib.reload(onboarding_task)
linkedin_runtime = importlib.reload(linkedin_runtime)
query_planner_runtime = importlib.reload(query_planner_runtime)

from browser.driver import build_driver
from core.config import load_app_config

app_config = load_app_config(verbose=False)
runtime_context = build_candidate_pipeline_context(ROOT, app_config)
browser_cfg = runtime_context["browser_cfg"]
task_config = runtime_context["task_config"]
llm_backend_name = runtime_context["llm_backend_name"]
llm_backend_settings = runtime_context["llm_backend_settings"]
llm_api_key_path = runtime_context["llm_api_key_path"]
llm_model = runtime_context["llm_model"]
llm_base_url = runtime_context["llm_base_url"]
llm_reasoning_effort = runtime_context["llm_reasoning_effort"]
llm_temperature = runtime_context["llm_temperature"]
browser_cfg["headless"] = False
browser_cfg["start_maximized"] = True

old_user_data_dir = Path(r"D:\_Desktop\Projects\Automations prj\User Data")
if old_user_data_dir.exists():
    browser_cfg["user_data_dir"] = str(old_user_data_dir)

store = embedded_mongo.EmbeddedMongoStore(ROOT / app_config["paths"]["mongo_file"])
driver = build_driver(browser_cfg)

onboarding_state_path = runtime_context["onboarding_state_path"]
candidate_scoring_state_path = runtime_context["candidate_scoring_state_path"]
detail_scoring_state_path = runtime_context["detail_scoring_state_path"]
artifact_root = runtime_context["artifact_root"]
candidate_artifact_dir = runtime_context["candidate_artifact_dir"]
detail_artifact_dir = runtime_context["detail_artifact_dir"]
query_plan_artifact_dir = runtime_context["query_plan_artifact_dir"]
query_search_artifact_dir = runtime_context["query_search_artifact_dir"]
candidate_artifact_dir.mkdir(parents=True, exist_ok=True)
detail_artifact_dir.mkdir(parents=True, exist_ok=True)
query_plan_artifact_dir.mkdir(parents=True, exist_ok=True)
query_search_artifact_dir.mkdir(parents=True, exist_ok=True)
linkedin_search_bundle = linkedin_runtime.load_instruction_bundle(ROOT / "agent-system-prompts" / "linkedin_search")
linkedin_query_planner_bundle = query_planner_runtime.load_planner_bundle(ROOT / "agent-system-prompts" / "linkedin_query_planner")


def _save_json(path: Path, payload: dict[str, object]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False)


def _load_json(path: Path) -> dict[str, object]:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def _canonicalize(value):
    if isinstance(value, dict):
        return {str(key): _canonicalize(value[key]) for key in sorted(value)}
    if isinstance(value, list):
        return [_canonicalize(item) for item in value]
    if isinstance(value, tuple):
        return [_canonicalize(item) for item in value]
    return value


def _parse_pages_spec(pages):
    if pages is None or pages == "":
        return [1]
    if isinstance(pages, int):
        return [pages] if pages > 0 else [1]
    if isinstance(pages, list):
        result = []
        for item in pages:
            if isinstance(item, int) and item > 0 and item not in result:
                result.append(item)
            elif isinstance(item, str):
                for page in _parse_pages_spec(item):
                    if page not in result:
                        result.append(page)
        return result or [1]
    if isinstance(pages, str):
        text = pages.strip()
        if not text:
            return [1]
        result = []
        for chunk in text.replace(" ", ",").split(","):
            piece = chunk.strip()
            if not piece:
                continue
            if "-" in piece:
                left, right = [part.strip() for part in piece.split("-", 1)]
                if left.isdigit() and right.isdigit():
                    start = int(left)
                    end = int(right)
                    if start > end:
                        start, end = end, start
                    for page in range(start, end + 1):
                        if page > 0 and page not in result:
                            result.append(page)
                    continue
            if piece.isdigit():
                page = int(piece)
                if page > 0 and page not in result:
                    result.append(page)
        return result or [1]
    try:
        page = int(str(pages).strip())
        return [page] if page > 0 else [1]
    except Exception:
        return [1]


def candidate_search_signature(candidate_search_input: dict[str, object]) -> str:
    normalized = {
        "keyword": candidate_search_input.get("keyword", ""),
        "location": candidate_search_input.get("location", ""),
        "filters": _canonicalize(candidate_search_input.get("filters", {})),
        "pages": _parse_pages_spec(candidate_search_input.get("pages")),
    }
    raw = json.dumps(normalized, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:16]


def candidate_artifact_path(candidate_search_input: dict[str, object]) -> Path:
    return candidate_artifact_dir / f"{candidate_search_signature(candidate_search_input)}.json"


def detail_artifact_path(listing_id: str) -> Path:
    return detail_artifact_dir / f"{str(listing_id).strip()}.json"


def load_or_fetch_candidate_result(candidate_search_input: dict[str, object]) -> tuple[dict[str, object], bool, Path]:
    path = candidate_artifact_path(candidate_search_input)
    if path.exists():
        payload = _load_json(path)
        return payload.get("candidate_result", {}), True, path
    candidate_result = linkedin.fetch_job_listings(
        driver,
        keyword=candidate_search_input["keyword"],
        location=candidate_search_input["location"],
        filters=candidate_search_input.get("filters"),
        pages=candidate_search_input.get("pages"),
        delays=candidate_search_input.get("delays"),
        verbose=bool(candidate_search_input.get("verbose", False)),
    )
    _save_json(
        path,
        {
            "candidate_search_input": candidate_search_input,
            "candidate_result": candidate_result,
        },
    )
    return candidate_result, False, path


def load_or_fetch_detail_result(candidate_result: dict[str, object], listing_id: str, detail_fetch_input: dict[str, object]) -> tuple[dict[str, object], bool, Path]:
    path = detail_artifact_path(listing_id)
    if path.exists():
        payload = _load_json(path)
        return payload.get("detail_result", {}), True, path
    detail_result = linkedin.fetch_listings_description(
        driver,
        candidate_result,
        listing_id=str(listing_id),
        delays=detail_fetch_input.get("delays"),
        verbose=bool(detail_fetch_input.get("verbose", False)),
    )
    _save_json(
        path,
        {
            "listing_id": str(listing_id),
            "detail_fetch_input": detail_fetch_input,
            "detail_result": detail_result,
        },
    )
    return detail_result, False, path


runtime_state = {
    "root": ROOT,
    "driver": driver,
    "store": store,
    "browser_cfg": browser_cfg,
    "onboarding_state_path": onboarding_state_path,
    "candidate_scoring_state_path": candidate_scoring_state_path,
    "detail_scoring_state_path": detail_scoring_state_path,
    "candidate_artifact_dir": candidate_artifact_dir,
    "detail_artifact_dir": detail_artifact_dir,
    "query_plan_artifact_dir": query_plan_artifact_dir,
    "query_search_artifact_dir": query_search_artifact_dir,
    "linkedin_search_bundle": linkedin_search_bundle,
    "linkedin_query_planner_bundle": linkedin_query_planner_bundle,
    "llm_backend_name": llm_backend_name,
    "llm_backend_settings": llm_backend_settings,
    "llm_api_key_path": llm_api_key_path,
    "llm_model": llm_model,
    "llm_base_url": llm_base_url,
    "llm_reasoning_effort": llm_reasoning_effort,
    "llm_temperature": llm_temperature,
    "task_config": task_config,
    "onboarding_result": {},
    "candidate_result": {},
    "scoring_result": {},
    "detail_fetch_result": {},
    "detail_scoring_result": {},
    "candidate_search_input": {},
    "candidate_scoring_input": {},
    "detail_fetch_input": {},
    "detail_scoring_input": {},
}

print(json.dumps({
    "root": str(ROOT),
    "browser_ready": True,
    "llm_backend": llm_backend_name,
}, indent=2, ensure_ascii=False))



def _compact_rows(rows):
    compact = []
    for row in list(rows or []):
        if isinstance(row, dict):
            compact.append({
                key: row.get(key)
                for key in (
                    "company",
                    "title",
                    "listing_id",
                    "job_id",
                    "location",
                    "decision",
                    "score",
                    "exclude_reason_code",
                    "exclude_reason_text",
                    "exclude_reason_current",
                    "exclude_reason_target",
                    "exclude_reason",
                )
                if key in row
            })
        else:
            compact.append(row)
    return compact


def _compact_batches(batches):
    return [
        {
            "batch_index": batch.get("batch_index"),
            "status": batch.get("status"),
            "llm_error": batch.get("llm_error"),
            "llm_response": batch.get("llm_response") if batch.get("llm_error") else "",
        }
        for batch in list(batches or [])
    ]


def _compact_summary(result, *, total_key, kept_key, excluded_key, next_key, reason_key, extra=None):
    summary = result.get("summary", {}) if isinstance(result, dict) else {}
    payload = {
        "status": result.get("status") if isinstance(result, dict) else None,
        total_key: summary.get(total_key),
        kept_key: summary.get(kept_key),
        excluded_key: summary.get(excluded_key),
        next_key: len(result.get(next_key, [])) if isinstance(result, dict) else None,
        reason_key: summary.get(reason_key, {}),
    }
    if extra:
        payload.update(extra)
    return payload


In [ ]:
# Cell 2: run onboarding digitization.
import importlib
import json
from core.config import load_app_config
from shared.runtime_config import build_candidate_pipeline_context
import tasks.onboarding_task as onboarding_task

app_config = load_app_config(verbose=False)
runtime_context = build_candidate_pipeline_context(ROOT, app_config)
task_config = runtime_context["task_config"]
onboarding_task = importlib.reload(onboarding_task)

profile_number = "1"
profiles_root = ROOT / "profiles"
selected_profile_dir = profiles_root / profile_number

onboarding_input = {
    "task_name": "onboarding_profile_digitization",
    "task_id": "onboarding-stage-001",
    "documents": [str(selected_profile_dir)],
}

onboarding_result = onboarding_task.run_onboarding_task(
    onboarding_input,
    state_path=runtime_state["onboarding_state_path"],
    heartbeat_seconds=1.0,
    step_delay_seconds=0.5,
    verbose=False,
)

digitized_user_handoff = onboarding_result.get("digitized_user", {})
runtime_state["onboarding_result"] = onboarding_result
runtime_state["digitized_user_handoff"] = digitized_user_handoff

print(json.dumps(digitized_user_handoff, indent=2, ensure_ascii=False))


In [ ]:
# Cell 3: generate clustered LinkedIn search queries from the digitized profile.
import importlib
import json
from core.config import load_app_config
from shared.runtime_config import build_candidate_pipeline_context
import tests.llm_test_linkedin.query_planner_runtime as query_planner_runtime

app_config = load_app_config(verbose=False)
runtime_context = build_candidate_pipeline_context(ROOT, app_config)
task_config = runtime_context["task_config"]
query_planner_runtime = importlib.reload(query_planner_runtime)

query_planner_task = runtime_state.get("query_planner_task") or (
    "Generate a small set of high-yield LinkedIn search clusters for this digitized user. "
    "Cover adjacent valid roles, keep overlap low, and avoid unrelated noise."
)

query_plan_result, reused_query_plan_artifact, query_plan_artifact_file = query_planner_runtime.load_or_fetch_query_plan(
    task=query_planner_task,
    digitized_user=digitized_user_handoff,
    artifact_root=query_plan_artifact_dir,
    bundle=linkedin_query_planner_bundle,
    llm_config=task_config["query_generation"]["llm"],
)

runtime_state["query_planner_task"] = query_planner_task
runtime_state["query_plan_result"] = query_plan_result
runtime_state["query_plan_input"] = {
    "task": query_planner_task,
    "digitized_user": digitized_user_handoff,
}

print(json.dumps({
    "status": query_plan_result.get("status", "success"),
    "artifact_reused": reused_query_plan_artifact,
    "artifact_path": str(query_plan_artifact_file),
    "search_context": query_plan_result.get("search_context", {}),
    "role_query_count": len(query_plan_result.get("role_queries", [])),
}, indent=2, ensure_ascii=False))
print(query_planner_runtime.compact_query_plan(query_plan_result))


In [ ]:
# Cell 4: execute the clustered LinkedIn searches.
import importlib
import json
from core.config import load_app_config
from shared.runtime_config import build_candidate_pipeline_context
import browser.linkedin as linkedin
import tests.llm_test_linkedin.query_planner_runtime as query_planner_runtime

app_config = load_app_config(verbose=False)
runtime_context = build_candidate_pipeline_context(ROOT, app_config)
task_config = runtime_context["task_config"]
linkedin = importlib.reload(linkedin)
query_planner_runtime = importlib.reload(query_planner_runtime)

search_result, reused_query_search_artifact, query_search_artifact_file = query_planner_runtime.run_query_plan_search(
    driver=driver,
    query_plan_result=query_plan_result,
    artifact_root=query_search_artifact_dir,
    linkedin_module=linkedin,
    search_config=task_config["linkedin_search"],
    verbose=task_config["linkedin_search"].get("verbose", False),
)

candidate_result = search_result
runtime_state["candidate_result"] = candidate_result
runtime_state["query_search_result"] = search_result
runtime_state["candidate_search_input"] = {
    "keyword": query_plan_result.get("search_context", {}).get("keyword", ""),
    "location": query_plan_result.get("search_context", {}).get("location", ""),
    "filters": query_plan_result.get("search_context", {}).get("filters", {}),
    "pages": task_config["linkedin_search"].get("max_page_count", 3),
    "search_task_id": candidate_result.get("search_task", {}).get("id", ""),
    "search_context": query_plan_result.get("search_context", {}),
    "query_plan": query_plan_result,
    "query_runs": candidate_result.get("query_runs", []),
}

print(json.dumps({
    "status": candidate_result.get("status"),
    "artifact_reused": reused_query_search_artifact,
    "artifact_path": str(query_search_artifact_file),
    "query_count": candidate_result.get("query_count"),
    "listing_count": len(candidate_result.get("listings", [])),
    "search_task_id": candidate_result.get("search_task", {}).get("id", ""),
    "pages_fetched": candidate_result.get("search_task", {}).get("pages_fetched", []),
    "visible_unfetched_pages": candidate_result.get("search_task", {}).get("visible_unfetched_pages", []),
    "query_runs": candidate_result.get("query_runs", []),
}, indent=2, ensure_ascii=False))

print(json.dumps(_compact_rows(candidate_result.get("listings", [])), indent=2, ensure_ascii=False))


In [ ]:
# Cell 5: score candidate listings for the next stage.
import importlib
import json
from core.config import load_app_config
from shared.runtime_config import build_candidate_pipeline_context
import tasks.candidate_scoring_task as candidate_scoring_task

app_config = load_app_config(verbose=False)
runtime_context = build_candidate_pipeline_context(ROOT, app_config)
task_config = runtime_context["task_config"]
candidate_scoring_task = importlib.reload(candidate_scoring_task)

candidate_scoring_input = {
    "task_name": "candidate_listing_scoring",
    "task_id": "candidate-scoring-stage-001",
    "digitized_user": digitized_user_handoff,
    "candidates": candidate_result.get("listings", []),
    "candidate_search_input": runtime_state.get("candidate_search_input", {}),
    "search_context": query_plan_result.get("search_context", {}),
    "search_task": candidate_result.get("search_task", {}),
    "batch_size": task_config["candidate_scoring"]["batch_size"],
}

scoring_result = candidate_scoring_task.run_candidate_scoring_task(
    candidate_scoring_input,
    state_path=runtime_state["candidate_scoring_state_path"],
    heartbeat_seconds=task_config["candidate_scoring"]["heartbeat_seconds"],
    step_delay_seconds=task_config["candidate_scoring"]["step_delay_seconds"],
    verbose=task_config["candidate_scoring"]["verbose"],
)

runtime_state["candidate_scoring_input"] = candidate_scoring_input
runtime_state["scoring_result"] = scoring_result

preview = [
    {
        "company": row.get("company"),
        "title": row.get("title"),
        "listing_id": row.get("listing_id"),
        "decision": row.get("decision"),
        "score": row.get("score"),
        "current": row.get("exclude_reason_current", ""),
        "target": row.get("exclude_reason_target", ""),
        "reason_code": row.get("exclude_reason_code"),
    }
    for row in scoring_result.get("scored_candidates", [])
]

print(json.dumps({
    "status": scoring_result.get("status"),
    "search_context": scoring_result.get("input", {}).get("search_context", {}),
    "total_candidates": scoring_result.get("summary", {}).get("total_candidates"),
    "batch_count": scoring_result.get("summary", {}).get("batch_count"),
    "kept_count": scoring_result.get("summary", {}).get("kept_count"),
    "excluded_count": scoring_result.get("summary", {}).get("excluded_count"),
    "reason_histogram": scoring_result.get("summary", {}).get("reason_histogram", {}),
    "next_stage_candidate_count": len(scoring_result.get("next_stage_candidates", [])),
    "preview": preview,
}, indent=2, ensure_ascii=False))

print(json.dumps(_compact_batches(scoring_result.get("batches", [])), indent=2, ensure_ascii=False))
print(json.dumps(_compact_rows(scoring_result.get("excluded_candidates", [])), indent=2, ensure_ascii=False))


In [ ]:
# Cell 6: fetch full detail rows for every non-excluded candidate, reusing notebook artifacts when available.
import importlib
import json
from core.config import load_app_config
from shared.runtime_config import build_candidate_pipeline_context
import browser.linkedin as linkedin

app_config = load_app_config(verbose=False)
runtime_context = build_candidate_pipeline_context(ROOT, app_config)
task_config = runtime_context["task_config"]
linkedin = importlib.reload(linkedin)

non_excluded_candidates = scoring_result.get("next_stage_candidates", [])

detail_fetch_input = task_config["detail_fetch"]

fetched_detail_rows = []
detail_fetch_artifacts = []
artifact_reuse_count = 0
for candidate in non_excluded_candidates:
    listing_id = str(candidate.get("listing_id") or candidate.get("job_id") or "").strip()
    if not listing_id:
        continue
    detail_result, reused_detail_artifact, detail_artifact_file = load_or_fetch_detail_result(
        candidate_result,
        listing_id,
        detail_fetch_input,
    )
    artifact_reuse_count += int(bool(reused_detail_artifact))
    detail_fetch_artifacts.append({
        "listing_id": listing_id,
        "artifact_reused": reused_detail_artifact,
        "artifact_path": str(detail_artifact_file),
    })
    for item in detail_result.get("ai", []):
        merged_row = {
            "listing": item.get("listing", {}),
            "detail": item.get("detail", {}),
            "company_profile": item.get("company_profile", {}),
            "dev": detail_result.get("dev", {}),
        }
        merged_row["listing_id"] = listing_id
        fetched_detail_rows.append(merged_row)

runtime_state["detail_fetch_input"] = detail_fetch_input
runtime_state["detail_fetch_result"] = {
    "rows": fetched_detail_rows,
    "artifacts": detail_fetch_artifacts,
    "artifact_reuse_count": artifact_reuse_count,
}

print(json.dumps({
    "candidate_count": len(non_excluded_candidates),
    "detail_row_count": len(fetched_detail_rows),
    "artifact_reuse_count": artifact_reuse_count,
    "artifacts": detail_fetch_artifacts,
}, indent=2, ensure_ascii=False))

print(json.dumps(_compact_rows(fetched_detail_rows), indent=2, ensure_ascii=False))


In [ ]:
# Cell 7: score the fetched job details.
import importlib
import json
from core.config import load_app_config
from shared.runtime_config import build_candidate_pipeline_context
import tasks.detail_scoring_task as detail_scoring_task

app_config = load_app_config(verbose=False)
runtime_context = build_candidate_pipeline_context(ROOT, app_config)
task_config = runtime_context["task_config"]
detail_scoring_task = importlib.reload(detail_scoring_task)

detail_scoring_input = {
    "task_name": "detail_listing_scoring",
    "task_id": "detail-scoring-stage-001",
    "digitized_user": digitized_user_handoff,
    "detail_rows": fetched_detail_rows,
    "batch_size": task_config["detail_scoring"]["batch_size"],
}

detail_scoring_result = detail_scoring_task.run_detail_scoring_task(
    detail_scoring_input,
    state_path=runtime_state["detail_scoring_state_path"],
    heartbeat_seconds=task_config["detail_scoring"]["heartbeat_seconds"],
    step_delay_seconds=task_config["detail_scoring"]["step_delay_seconds"],
    verbose=task_config["detail_scoring"]["verbose"],
)

runtime_state["detail_scoring_input"] = detail_scoring_input
runtime_state["detail_scoring_result"] = detail_scoring_result

print(json.dumps({
    "status": detail_scoring_result.get("status"),
    "total_rows": detail_scoring_result.get("summary", {}).get("total_rows"),
    "batch_count": detail_scoring_result.get("summary", {}).get("batch_count"),
    "kept_count": detail_scoring_result.get("summary", {}).get("kept_count"),
    "excluded_count": detail_scoring_result.get("summary", {}).get("excluded_count"),
    "reason_histogram": detail_scoring_result.get("summary", {}).get("reason_histogram", {}),
    "section_histogram": detail_scoring_result.get("summary", {}).get("section_histogram", {}),
    "next_stage_row_count": len(detail_scoring_result.get("next_stage_rows", [])),
}, indent=2, ensure_ascii=False))

print(json.dumps(_compact_batches(detail_scoring_result.get("batches", [])), indent=2, ensure_ascii=False))
print(json.dumps(_compact_rows(detail_scoring_result.get("excluded_detail_rows", [])), indent=2, ensure_ascii=False))
print(json.dumps(_compact_rows(detail_scoring_result.get("kept_detail_rows", [])), indent=2, ensure_ascii=False))
print(json.dumps(_compact_rows(detail_scoring_result.get("scored_detail_rows", [])), indent=2, ensure_ascii=False))
